Nama Anggota :
1. Ammalya Khoirun Nizak (25031554036)
2. Muhammad Rifky Ihwan (25031554241)

In [ ]:
!pip install demoji
!pip install Levenshtein
!pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.3 MB/s eta 0:00:00
  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-h6zcwybr
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-h6zcwybr
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done
  Created wheel for mpstemmer: filename=mpstemmer-0.1.0-py3-none-any.whl size=99832 sha256=0cc573b30c404c0ef25571ddbb142834fd676c483b23d5b5d621a70ce461207b
  Stored in directory: /tmp/pip-ephem-wheel-cache-prp70xvn/wheels/d0/b9/fd/dba0c7d44e5338db51da6ffcbe96db4326db3996a403f5ced1
Successfully built mpstemmer


In [ ]:
import pandas as pd

data = {'Id' : [1],
        'Username' : ['@jenostalker'],
        'text' : ['BERTAHAN HIDUP SAMPE MAU MENINGGAL GITU DIBILANG NORMAL EMG DASAR MANUSIA GATAU DIRI. GUYS DONT STOP TALKING ABOUT KALIMANTAN... 😭 😭 😭 KABUTNYA SEPERTI APA? SEPERTI INI, INI LUAR BIASA DZOLIM. SEPERTI INI  YANG KATANYA NORMAL?? TURUN LOKASI, TURUN MAKANYA JGN CUMA "SAYA DPT LAPORAN. #PALANGKARAYA #PRAYFORLAMPUNG #PRAYFORSULAWESI #PRAYFORKALIMANTAN #PRAYFORSUMATERA #PRAYFORACEH #PRAYFORNTT #PRAYFORNTB #PRAYFORPAPUA #PRAYFORINDONESIA #AllEyesOnIndonesia"']}

df = pd.DataFrame(data)
df


,Id,Username,text
0,1,@jenostalker,BERTAHAN HIDUP SAMPE MAU MENINGGAL GITU DIBILA...


In [ ]:
import re
import demoji
import nltk
from nltk.corpus import stopwords
from mpstemmer import MPStemmer

nltk.download('stopwords')
stemmer = MPStemmer()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Mengubah semua huruf kapital menjadi huruf kecil

In [ ]:
df['lower_case'] = df['text'].str.lower()
df[['text', 'lower_case']]

,text,lower_case
0,BERTAHAN HIDUP SAMPE MAU MENINGGAL GITU DIBILA...,bertahan hidup sampe mau meninggal gitu dibila...


Menghapus seluruh hastag

In [ ]:
def remove_hastag(text):
  return re.sub(r'#\w+', '', text)

df['rm_hastag'] = df['lower_case'].apply(remove_hastag)
df[['lower_case', 'rm_hastag']]

,lower_case,rm_hastag
0,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampe mau meninggal gitu dibila...


Menghapus ikon emoji

In [ ]:
def remove_emoji(text):
  return demoji.replace(text, repl="")

df['rm_emoji'] = df['rm_hastag'].apply(remove_emoji)
df[['rm_hastag', 'rm_emoji']]

,rm_hastag,rm_emoji
0,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampe mau meninggal gitu dibila...


Menghapus titik, tanda tanda, tanda petik dan spasi ganda

In [ ]:
def remove_punctuation(text):
  text = re.sub(r'[!"#$%&\'()*+,\-./:;<=>?@[\]^_`{|}~“”]', '', text)
  text = re.sub(r'\s+', ' ', text).strip()
  return text

df['rm_punc'] = df['rm_emoji'].apply(remove_punctuation)
df[['rm_emoji', 'rm_punc']]

,rm_emoji,rm_punc
0,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampe mau meninggal gitu dibila...


Normalisasi kata gaul dan singkatan menjadi kata baku

In [ ]:
dict_alay = {'sampe': 'sampai',
              'emg': 'memang',
              'dont': 'jangan',
              'jgn': 'jangan',
              'dpt': 'dapat',
              'guys': 'teman-teman'}

def normalize_text(text):
  words = text.split()
  result = [dict_alay.get(w, w) for w in words]
  return ' '.join(result)

df['norm_slang'] = df['rm_punc'].apply(normalize_text)
df[['rm_punc', 'norm_slang']]

,rm_punc,norm_slang
0,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampai mau meninggal gitu dibil...


Mengubah kata berimbuhan menjadi kata dasar (Stemming)

In [ ]:
df['stem'] = df['norm_slang'].apply(lambda x : stemmer.stem_kalimat(x) if isinstance(x, str) else '')
df[['norm_slang', 'stem']]

,norm_slang,stem
0,bertahan hidup sampai mau meninggal gitu dibil...,tahan hidup sampai mau tinggal gitu bilang nor...


menghapus kata-kata umum yang kurang bernilai analisis

In [ ]:
sw = set(stopwords.words('indonesian'))

def remove_stopwords(text):
  return " ".join([word for word in str(text).split() if word not in sw])

df['prepro'] = df['stem'].apply(remove_stopwords)
df[['stem', 'prepro']]

,stem,prepro
0,tahan hidup sampai mau tinggal gitu bilang nor...,tahan hidup tinggal gitu bilang normal dasar m...


In [ ]:
print("Teks Asli :")
print(df['text'].iloc[0])
print("\nHasil pre-processing:")
print(df['prepro'].iloc[0])

Teks Asli :
BERTAHAN HIDUP SAMPE MAU MENINGGAL GITU DIBILANG NORMAL EMG DASAR MANUSIA GATAU DIRI. GUYS DONT STOP TALKING ABOUT KALIMANTAN... 😭 😭 😭 KABUTNYA SEPERTI APA? SEPERTI INI, INI LUAR BIASA DZOLIM. SEPERTI INI  YANG KATANYA NORMAL?? TURUN LOKASI, TURUN MAKANYA JGN CUMA "SAYA DPT LAPORAN. #PALANGKARAYA #PRAYFORLAMPUNG #PRAYFORSULAWESI #PRAYFORKALIMANTAN #PRAYFORSUMATERA #PRAYFORACEH #PRAYFORNTT #PRAYFORNTB #PRAYFORPAPUA #PRAYFORINDONESIA #AllEyesOnIndonesia"

Hasil pre-processing:
tahan hidup tinggal gitu bilang normal dasar manusia gatau teman-teman stop talking about kalimantan kabut dzolim normal turun lokasi turun lapor


**Comment Youtube**

In [ ]:
def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', str(text))

def remove_mention(text):
    return re.sub(r'@\w+', '', str(text))

if 'lower_case' not in df.columns:
    df['lower_case'] = df['text'].str.lower()

df['rm_url'] = df['lower_case'].apply(remove_url)
df['rm_mention'] = df['rm_url'].apply(remove_mention)
df[['lower_case', 'rm_url', 'rm_mention']].head()

,lower_case,rm_url,rm_mention
0,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampe mau meninggal gitu dibila...,bertahan hidup sampe mau meninggal gitu dibila...
